In [3]:
COLLECTION_NAME = "HYBRID_COLLECTION"

In [4]:
import json
from groq_helper import *
from pprint import pprint

In [5]:
from qdrant_client import QdrantClient, models
import os

client = QdrantClient(url = os.getenv("QDRANT_URL"),
                      api_key=os.getenv("QDRANT_KEY"),timeout=600)

c:\Ambarish\AWSBEDROCK-KOLKATA-COMMUNITYDAY-2025\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
client.count(COLLECTION_NAME)

CountResult(count=49)

In [7]:
from fastembed import TextEmbedding, SparseTextEmbedding, LateInteractionTextEmbedding

dense_embedding_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2")
bm25_embedding_model = SparseTextEmbedding("Qdrant/bm25")
late_interaction_embedding_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]c:\Ambarish\AWSBEDROCK-KOLKATA-COMMUNITYDAY-2025\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ambarish Ganguly\AppData\Local\Temp\fastembed_cache\models--qdrant--all-MiniLM-L6-v2-onnx. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetchin

In [8]:
query_text = "Describe about protons in the stroma"

### Dense Embedding 

In [9]:
results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=next(dense_embedding_model.query_embed(query_text)),
    using="all-MiniLM-L6-v2",
    limit=10,
    with_payload=True,
)

In [10]:
def get_response(results,query_text):
    # Assuming results is the output from client.query_points
    results_to_return = []

# Extracting text from results
    for point in results.points:
        results_to_return.append(point.payload["content"])
    context = "\n".join(results_to_return)
    # answer = generate_reply_from_context(query, context,[])
    response_text = generate_reply_from_context(query_text,context,[])

    return response_text

In [11]:
pprint(get_response(results,query_text))

('In the context of photosynthesis, the stroma is the fluid-filled region of a '
 'chloroplast where the Calvin cycle takes place. The stroma is the site where '
 'protons (H+ ions) play a crucial role in the synthesis of ATP and NADPH.\n'
 '\n'
 '**Proton gradient and ATP synthesis**\n'
 '\n'
 'During the light-dependent reactions of photosynthesis, protons are pumped '
 'across the thylakoid membrane into the thylakoid lumen, creating a proton '
 'gradient. This gradient has a high concentration of protons in the thylakoid '
 'lumen and a low concentration in the stroma.\n'
 '\n'
 'The movement of protons back across the membrane, from the thylakoid lumen '
 'to the stroma, drives the synthesis of ATP through the process of '
 'chemiosmosis. This is because the proton gradient has a high energy '
 'potential, which is harnessed by the enzyme ATP synthase to produce ATP from '
 'ADP and Pi.\n'
 '\n'
 '**Proton concentration in the stroma**\n'
 '\n'
 'In the stroma, the proton concentr

### Sparse Embedding

In [12]:
sparse_query_vector = next(bm25_embedding_model.query_embed(query_text))

In [13]:
late_query_vector = next(late_interaction_embedding_model.query_embed(query_text))

In [14]:
results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=models.SparseVector(**sparse_query_vector.as_object()),
        using="bm25",
        with_payload=True,
        limit=10,
    )

In [15]:
pprint(get_response(results,query_text))

('According to the text, within the chloroplast, protons in the stroma '
 'decrease in number. This happens because:\n'
 '\n'
 '* As electrons move through the photosystems, protons are transported across '
 'the membrane.\n'
 '* The primary acceptor of electrons, which is located towards the outer side '
 'of the membrane, transfers its electron to an H+ carrier. This molecule '
 'removes a proton from the stroma while transporting an electron.\n'
 '* The NADP reductase enzyme, located on the stroma side of the membrane, '
 'uses protons to reduce NADP+ to NADPH + H+. These protons are also removed '
 'from the stroma.\n'
 '\n'
 'As a result, there is a decrease in the number of protons in the stroma, '
 'while there is an accumulation of protons in the lumen (the inner side of '
 'the thylakoid membrane). This creates a proton gradient across the thylakoid '
 'membrane, which is important for the synthesis of ATP. The breakdown of this '
 'gradient, due to the movement of protons bac

### RRF

In [16]:
prefetch = [
        models.Prefetch(
            query=next(dense_embedding_model.query_embed(query_text)),
            using="all-MiniLM-L6-v2",
            limit=20,
        ),
        models.Prefetch(
            query=models.SparseVector(**sparse_query_vector.as_object()),
            using="bm25",
            limit=20,
        ),
    ]
results = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=prefetch,
        query=models.FusionQuery(
            fusion=models.Fusion.RRF,
        ),
        with_payload=True,
        limit=10,
    )

In [17]:
pprint(get_response(results,query_text))

('In the context of photosynthesis, the stroma is the fluid-filled region of a '
 'chloroplast where the Calvin cycle takes place. The stroma is the site where '
 'carbon dioxide is fixed into organic molecules using the energy from ATP and '
 'NADPH produced in the light-dependent reactions.\n'
 '\n'
 'Protons (H+ ions) play a crucial role in the process of photosynthesis, and '
 'their movement across the thylakoid membrane is essential for the generation '
 'of ATP. In the stroma, protons are involved in the following processes:\n'
 '\n'
 '1. **Proton removal**: During the light-dependent reactions, protons are '
 'pumped across the thylakoid membrane into the lumen, creating a proton '
 'gradient. As a result, the stroma becomes depleted of protons, and the pH of '
 'the stroma increases.\n'
 '2. **Proton utilization**: In the stroma, protons are used for the reduction '
 'of NADP+ to NADPH by the enzyme NADP reductase. This reaction requires '
 'protons to proceed, and the protons

### Late Interaction Fusion

In [18]:
prefetch = [
        models.Prefetch(
            query=next(dense_embedding_model.query_embed(query_text)),
            using="all-MiniLM-L6-v2",
            limit=20,
        ),
        models.Prefetch(
            query=models.SparseVector(**sparse_query_vector.as_object()),
            using="bm25",
            limit=20,
        ),
    ]
results = client.query_points(
        collection_name=COLLECTION_NAME,
    prefetch=prefetch,
    query=late_query_vector,
    using="colbertv2.0",
    with_payload=True,
    limit=5,
)

In [19]:
pprint(get_response(results,query_text))

('In the context of photosynthesis, the stroma is the fluid-filled region of a '
 'chloroplast where the Calvin cycle takes place. Protons (H+ ions) play a '
 'crucial role in the light-dependent reactions of photosynthesis, which occur '
 'in the thylakoid membranes.\n'
 '\n'
 'During the light-dependent reactions, protons are pumped across the '
 'thylakoid membrane into the thylakoid lumen, creating a proton gradient. '
 'This gradient has two effects:\n'
 '\n'
 '1. **Proton accumulation in the lumen**: The protons accumulate in the '
 'thylakoid lumen, increasing the concentration of H+ ions and decreasing the '
 'pH.\n'
 '2. **Proton decrease in the stroma**: As protons are pumped into the lumen, '
 'the concentration of H+ ions in the stroma decreases. This creates a proton '
 'gradient across the thylakoid membrane, with a higher concentration of '
 'protons in the lumen and a lower concentration in the stroma.\n'
 '\n'
 'The decrease in proton concentration in the stroma is imp